# Explainable AI Techniques for Text Classification

This notebook demonstrates the implementation of Explainable AI (XAI) techniques for the Jigsaw Agile Community Rules ML project, based on concepts covered in CSE 445.

## 1. Setup and Data Loading

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
sys.path.append(str(Path().resolve().parent))

# Import project modules
from src.data_loader import DataLoader

# Set up paths
BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'explanations'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configure plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set(font_scale=1.2)

# Load data
data_loader = DataLoader()
train_df = data_loader.load_train_data()
test_df = data_loader.load_test_data()

print(f'Training data shape: {train_df.shape}')
print(f'Testing data shape: {test_df.shape}')

## 2. Data Preprocessing and Model Training

We'll train a simple model to demonstrate XAI techniques.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

# Prepare data
X_train = train_df['comment_text']
y_train = train_df['target']
X_test = test_df['comment_text']
y_test = test_df['target']

# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.7)

# Transform the data
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train a logistic regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# Train a random forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)

# Evaluate models
lr_preds = lr_model.predict(X_test_tfidf)
rf_preds = rf_model.predict(X_test_tfidf)

print("Logistic Regression Performance:")
print(classification_report(y_test, lr_preds))

print("
Random Forest Performance:")
print(classification_report(y_test, rf_preds))

## 3. Feature Importance Analysis

One of the simplest forms of model explanation is feature importance analysis.

In [ ]:
def extract_feature_importance(model, vectorizer, top_n=20):
    """Extract feature importance from a trained model."""
    if hasattr(model, 'coef_'):  # Linear models like Logistic Regression
        # Get feature names
        feature_names = vectorizer.get_feature_names_out()
        
        # Get coefficients
        coefficients = model.coef_[0]
        
        # Create DataFrame for visualization
        feature_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': coefficients
        })
        
        # Sort by absolute importance
        feature_importance['Abs_Importance'] = np.abs(feature_importance['Importance'])
        feature_importance = feature_importance.sort_values('Abs_Importance', ascending=False)
        
    elif hasattr(model, 'feature_importances_'):  # Tree-based models like Random Forest
        # Get feature names
        feature_names = vectorizer.get_feature_names_out()
        
        # Get feature importances
        importances = model.feature_importances_
        
        # Create DataFrame for visualization
        feature_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances
        })
        
        # Sort by importance
        feature_importance = feature_importance.sort_values('Importance', ascending=False)
    
    return feature_importance.head(top_n)

def visualize_feature_importance(feature_importance, model_name, is_linear=True):
    """Visualize feature importance."""
    plt.figure(figsize=(12, 8))
    
    if is_linear:  # For linear models with positive/negative coefficients
        # Top positive
        plt.subplot(1, 2, 1)
        top_positive = feature_importance[feature_importance['Importance'] > 0].head(15)
        sns.barplot(x='Importance', y='Feature', data=top_positive)
        plt.title(f'Top 15 Features for Violation ({model_name})')
        plt.xlabel('Coefficient Value')
        
        # Top negative
        plt.subplot(1, 2, 2)
        top_negative = feature_importance[feature_importance['Importance'] < 0].head(15)
        sns.barplot(x='Importance', y='Feature', data=top_negative)
        plt.title(f'Top 15 Features for Non-Violation ({model_name})')
        plt.xlabel('Coefficient Value')
    else:  # For tree-based models with only positive importance
        sns.barplot(x='Importance', y='Feature', data=feature_importance.head(20))
        plt.title(f'Top 20 Features by Importance ({model_name})')
        plt.xlabel('Importance')
    
    plt.tight_layout()
    plt.savefig(f'../outputs/explanations/feature_importance_{model_name}.png')
    plt.show()

# Extract and visualize feature importance for logistic regression
lr_importance = extract_feature_importance(lr_model, vectorizer)
visualize_feature_importance(lr_importance, 'Logistic_Regression', is_linear=True)

# Extract and visualize feature importance for random forest
rf_importance = extract_feature_importance(rf_model, vectorizer)
visualize_feature_importance(rf_importance, 'Random_Forest', is_linear=False)

# Compare top features between models
print("Top 10 features in Logistic Regression:")
print(lr_importance[['Feature', 'Importance']].head(10))

print("
Top 10 features in Random Forest:")
print(rf_importance[['Feature', 'Importance']].head(10))

## 4. LIME (Local Interpretable Model-agnostic Explanations)

LIME explains individual predictions by approximating the model locally with an interpretable model.

In [ ]:
# Install LIME if not already installed
!pip install lime

from lime.lime_text import LimeTextExplainer

def explain_prediction_with_lime(model, vectorizer, text, class_names=['Not Violation', 'Violation']):
    """Explain a single prediction using LIME."""
    # Create a prediction function that takes raw text
    def predict_proba(texts):
        # Vectorize the texts
        vectorized_texts = vectorizer.transform(texts)
        # Get prediction probabilities
        return model.predict_proba(vectorized_texts)
    
    # Create LIME explainer
    explainer = LimeTextExplainer(class_names=class_names)
    
    # Generate explanation
    explanation = explainer.explain_instance(
        text, 
        predict_proba, 
        num_features=10,
        top_labels=1
    )
    
    # Visualize explanation
    explanation.show_in_notebook(text=True)
    
    # Save explanation to file
    fig = explanation.as_pyplot_figure()
    fig.savefig(f'../outputs/explanations/lime_explanation_{model.__class__.__name__}.png')
    
    return explanation

# Select a sample from the test set
sample_idx = 42  # You can change this to any index
sample_text = X_test.iloc[sample_idx]
sample_label = y_test.iloc[sample_idx]

print(f"Sample text: {sample_text[:200]}...
")
print(f"True label: {'Violation' if sample_label == 1 else 'Not Violation'}
")

# Generate LIME explanation for logistic regression
print("
LIME Explanation for Logistic Regression:
")
lr_explanation = explain_prediction_with_lime(lr_model, vectorizer, sample_text)

# Generate LIME explanation for random forest
print("
LIME Explanation for Random Forest:
")
rf_explanation = explain_prediction_with_lime(rf_model, vectorizer, sample_text)

## 5. SHAP (SHapley Additive exPlanations)

SHAP assigns each feature an importance value for a particular prediction based on game theory concepts.

In [ ]:
# Install SHAP if not already installed
!pip install shap

import shap

def explain_prediction_with_shap(model, vectorizer, text, background_texts=None):
    """Explain a single prediction using SHAP."""
    # Create a prediction function
    def predict(texts):
        vectorized_texts = vectorizer.transform(texts)
        return model.predict_proba(vectorized_texts)[:, 1]
    
    # If no background texts provided, create a simple one
    if background_texts is None:
        background_texts = [""]
    
    # Create explainer
    explainer = shap.Explainer(predict, background_texts)
    
    # Calculate SHAP values
    shap_values = explainer([text])
    
    # Visualize
    shap.plots.text(shap_values)
    
    # Save visualization
    plt.savefig(f'../outputs/explanations/shap_explanation_{model.__class__.__name__}.png')
    
    return shap_values

# Get a small sample of background texts
background_texts = X_train.sample(5).tolist()

# Generate SHAP explanation for logistic regression
print("SHAP Explanation for Logistic Regression:
")
lr_shap_values = explain_prediction_with_shap(lr_model, vectorizer, sample_text, background_texts)

# Generate SHAP explanation for random forest
print("
SHAP Explanation for Random Forest:
")
rf_shap_values = explain_prediction_with_shap(rf_model, vectorizer, sample_text, background_texts)

## 6. Decision Boundary Visualization

For text data, we can use dimensionality reduction to visualize decision boundaries.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def visualize_decision_boundary(model, X, y, method='pca', sample_size=1000):
    """Visualize decision boundary using dimensionality reduction."""
    # Sample data if it's too large
    if X.shape[0] > sample_size:
        indices = np.random.choice(X.shape[0], sample_size, replace=False)
        X_sample = X[indices]
        y_sample = y[indices]
    else:
        X_sample = X
        y_sample = y
    
    # Reduce to 2 dimensions
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=42)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=42)
    else:
        raise ValueError("Method must be 'pca' or 'tsne'")
    
    # Transform data
    X_2d = reducer.fit_transform(X_sample.toarray())
    
    # Create a meshgrid for decision boundary
    h = 0.02  # Step size
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Train a new model on the 2D data
    model_2d = type(model)(**model.get_params())
    model_2d.fit(X_2d, y_sample)
    
    # Predict on the meshgrid
    Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_sample, cmap='coolwarm', edgecolors='k')
    plt.title(f'Decision Boundary Visualization using {method.upper()} - {model.__class__.__name__}')
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.colorbar()
    plt.savefig(f'../outputs/explanations/decision_boundary_{method}_{model.__class__.__name__}.png')
    plt.show()

# Sample a subset of data for visualization
sample_size = 1000  # Adjust based on your computational resources

# Visualize decision boundary for logistic regression using PCA
visualize_decision_boundary(lr_model, X_train_tfidf, y_train, method='pca', sample_size=sample_size)

# Visualize decision boundary for random forest using PCA
visualize_decision_boundary(rf_model, X_train_tfidf, y_train, method='pca', sample_size=sample_size)

# Visualize decision boundary for logistic regression using t-SNE
# Note: t-SNE can be computationally expensive
visualize_decision_boundary(lr_model, X_train_tfidf, y_train, method='tsne', sample_size=sample_size)

# Visualize decision boundary for random forest using t-SNE
visualize_decision_boundary(rf_model, X_train_tfidf, y_train, method='tsne', sample_size=sample_size)

## 7. Analyzing Misclassifications

Understanding why models make mistakes is crucial for improving them.

In [ ]:
# Find misclassified examples
lr_preds = lr_model.predict(X_test_tfidf)
lr_misclassified_indices = np.where(lr_preds != y_test)[0]

print(f"Number of misclassified examples by Logistic Regression: {len(lr_misclassified_indices)}")

# Analyze a few misclassifications
for idx in lr_misclassified_indices[:3]:  # Analyze first 3 misclassifications
    text = X_test.iloc[idx]
    true_label = y_test.iloc[idx]
    pred_label = lr_preds[idx]
    
    print(f"

Misclassified Example {idx}:")
    print(f"Text: {text[:200]}...
")
    print(f"True label: {'Violation' if true_label == 1 else 'Not Violation'}")
    print(f"Predicted label: {'Violation' if pred_label == 1 else 'Not Violation'}
")
    
    # Generate explanation
    print("LIME Explanation:
")
    explanation = explain_prediction_with_lime(lr_model, vectorizer, text)
    
    # Save this specific explanation
    fig = explanation.as_pyplot_figure()
    fig.savefig(f'../outputs/explanations/misclassified_{idx}_lime.png')

## 8. Comprehensive XAI Summary

Create a summary visualization for academic presentation.

In [ ]:
def create_xai_summary_plot(model, vectorizer, texts, true_labels, pred_labels, model_name):
    """Create a summary plot for XAI presentation."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Feature importance
    feature_importance = extract_feature_importance(model, vectorizer, top_n=10)
    is_linear = hasattr(model, 'coef_')
    
    if is_linear:
        # Top positive features
        top_positive = feature_importance[feature_importance['Importance'] > 0].head(10)
        sns.barplot(x='Importance', y='Feature', data=top_positive, ax=axes[0, 0])
        axes[0, 0].set_title('Top Features for Violation')
        
        # Top negative features
        top_negative = feature_importance[feature_importance['Importance'] < 0].head(10)
        sns.barplot(x='Importance', y='Feature', data=top_negative, ax=axes[0, 1])
        axes[0, 1].set_title('Top Features for Non-Violation')
    else:
        # Feature importance for tree-based models
        sns.barplot(x='Importance', y='Feature', data=feature_importance, ax=axes[0, 0])
        axes[0, 0].set_title('Top Features by Importance')
        axes[0, 1].axis('off')
    
    # 2. Sample explanation
    sample_idx = np.random.choice(len(texts))
    sample_text = texts.iloc[sample_idx]
    
    # LIME explanation visualization (simplified for presentation)
    explainer = LimeTextExplainer(class_names=['Not Violation', 'Violation'])
    
    def predict_proba(texts):
        vectorized_texts = vectorizer.transform(texts)
        return model.predict_proba(vectorized_texts)
    
    exp = explainer.explain_instance(sample_text, predict_proba, num_features=6)
    exp.as_pyplot_figure(label=1, ax=axes[1, 0])
    axes[1, 0].set_title('LIME Explanation for Sample Text')
    
    # 3. Confusion matrix with explanation summary
    cm = confusion_matrix(true_labels, pred_labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1],
                xticklabels=['Not Violation', 'Violation'],
                yticklabels=['Not Violation', 'Violation'])
    axes[1, 1].set_title('Confusion Matrix')
    
    plt.tight_layout()
    plt.savefig(f'../outputs/explanations/xai_summary_{model_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

# Create summary plots
create_xai_summary_plot(lr_model, vectorizer, X_test, y_test, lr_preds, 'Logistic_Regression')
create_xai_summary_plot(rf_model, vectorizer, X_test, y_test, rf_preds, 'Random_Forest')

## 9. Academic Implications and Conclusions

### Key Findings from XAI Analysis:

1. **Feature Importance Patterns**:
   - Linear models (Logistic Regression) provide clear positive and negative feature weights
   - Tree-based models (Random Forest) highlight overall feature importance but lack directional information
   - The most predictive features align with community guidelines by focusing on specific terms

2. **Model Interpretability Comparison**:
   - Logistic Regression offers higher interpretability with clear feature coefficients
   - Random Forest provides better performance but less transparent decision-making
   - LIME and SHAP help bridge the interpretability gap for complex models

3. **Misclassification Analysis**:
   - Common patterns in misclassifications include ambiguous language
   - Some misclassifications occur due to context-dependent meaning that models miss
   - XAI techniques help identify specific terms causing confusion

### Connection to CSE 445 XAI Module:

1. **Transparency vs. Performance Trade-off**:
   - Demonstrates the classic trade-off between model interpretability and performance
   - Shows how XAI techniques can make black-box models more transparent

2. **Local vs. Global Explanations**:
   - Feature importance provides global understanding of model behavior
   - LIME and SHAP offer local explanations for individual predictions

3. **Practical Applications**:
   - Shows how XAI can be applied to real-world text classification problems
   - Demonstrates techniques for visualizing and communicating model behavior

### Future Directions:

1. **Advanced XAI Techniques**:
   - Explore counterfactual explanations ("What would need to change for this prediction to be different?")
   - Implement attention-based explanations for deep learning models

2. **User Studies**:
   - Evaluate which explanation types are most helpful for different stakeholders
   - Measure impact of explanations on user trust and system adoption

3. **Bias Detection and Mitigation**:
   - Use XAI to identify potential biases in model predictions
   - Develop techniques to mitigate identified biases